# 6회차 · 내 문서로 답하는 챗봇 만들기(RAG 샌드박스)

오늘은 **내가 고른 자료**로 검색 기반 생성(RAG)을 직접 만들어 봄. 정해진 답을 따라 치는 실습이 아님.
자료를 넣고, 설정을 바꾸고, 결과가 어떻게 달라지는지 그림과 나란히 놓인 답으로 확인하며 **내 챗봇**을 완성함.

- 셀은 위에서 아래로 한 번씩 실행함(`Shift + Enter`). 조절판이 나오면 값을 바꾸고 단추를 누름
- 같은 조절판을 몇 번이든 다시 눌러도 됨. 바꾸는 값은 **한 번에 하나**만 바꿔야 무엇 때문에 달라졌는지 알 수 있음
- 내 자료는 서버의 `my/` 폴더에만 남음. 본인과 타인의 실제 개인정보, 외부 반출이 금지된 자료는 넣지 않음
- 끝낼 때 맨 아래 '결과 묶기'를 실행해 zip 파일을 내려받음

| 단계 | 하는 일 | 필수 |
|---|---|---|
| 0 | 시작 : GPU 와 내 프로젝트 상태 | 필수 |
| 1 | 모델 올리기(GPU 계기판) | 필수 |
| 2 | 내 재료 넣기 | 필수 |
| 3 | 내 확인 질문 만들고, 문서 없이 풀어 보기 | 필수 |
| 4 | 자르기 실험(청크) | 선택 |
| 5 | 의미 지도 | 선택 |
| 6 | 찾기 실험(검색) | 선택 |
| 7 | 문서를 붙여 답하기 | 필수 |
| 8 | 설정 실험 : 무엇을 바꾸면 좋아지는가 | 필수 |
| 9 | 모델 겨루기 | 선택 |
| 10 | 내 챗봇 완성 · 저장 | 필수 |
| 11 | 판단 기록 · 결과 묶기 | 필수 |

선택 단계(4 · 5 · 6 · 9)는 관심 가는 것을 **둘 이상** 고름.

## 0. 시작
아래 셀을 실행하면 이 서버의 GPU 와 내 프로젝트 상태가 보임. 수업 중에 노트북을 다시 열었을 때도 이 셀부터 실행함.

In [ ]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "lab.py").exists())
sys.path.insert(0, str(root / "src"))
import lab
lab.start(6)

## 1. 모델 올리기
언어 모델을 GPU 메모리에 올림. 올린 뒤 계기판에서 **메모리가 얼마나 찼는지**, 여유로 **무엇을 더 올릴 수 있는지** 확인함.
- 기본 모델은 `skt/A.X-4.0-Light`(한국어, 7B급)임. 먼저 이 모델을 올림
- 메모리가 넉넉한 서버는 모델을 여러 개 함께 올려 둘 수 있음. 9단계 '모델 겨루기'에서 씀

In [ ]:
lab.model_panel()

## 2. 내 재료 넣기
챗봇이 답의 근거로 쓸 문서를 넣음. 네 가지 방법을 섞어 써도 됨.
- **직접 쓰기** : 내가 지어낸 설정, 동아리 회칙, 가상의 가게 안내문처럼 모델이 알 수 없는 내용일수록 효과가 잘 보임
- **위키백과** : 관심 주제의 문서 제목을 적으면 가져옴(공개 자료, 출처 자동 기록)
- **예제 재료** : 무엇을 넣을지 정하지 못했으면 준비된 가상 자료 G · L · S 가운데 하나로 시작함
- **파일** : 내 PC 의 `.txt` · `.md` 파일

먼저 '내 주제'를 한 줄로 적어 저장하고, 문서를 **3건 이상** 넣음.

In [ ]:
lab.docs_panel()

## 3. 내 확인 질문
내 문서를 읽어야만 답할 수 있는 질문을 **5건 이상** 적음. 한 줄에 `질문 | 답에 꼭 들어가야 할 단어` 형식임.
기대 단어는 답이 맞았는지 자동으로 가려내는 데 씀(여러 개면 쉼표로 구분).

먼저 '둘 다'로 실행해, 문서 없이 풀었을 때와 문서를 붙여 풀었을 때의 점수를 비교함.
- 문서 없이 답한 내용을 읽어 봄. 모른다고 하는가, 그럴듯하게 지어내는가

In [ ]:
lab.quiz_panel()

## 4. 자르기 실험(선택)
문서는 통째로 넣지 않고 **청크**로 잘라 넣음. 방식과 크기를 바꾸면 경계가 어떻게 달라지는지 봄.
- 너무 작으면 표의 머리글과 숫자가 다른 청크로 갈라지고, 너무 크면 관련 없는 내용이 함께 딸려 옴
- 마음에 드는 설정이 나오면 '이 설정을 내 설정으로 저장'을 켜고 다시 누름

In [ ]:
lab.chunk_panel()

## 5. 의미 지도(선택)
청크 하나하나를 임베딩 모델이 벡터로 바꾼 뒤 2차원에 펼친 그림임. 가까이 있는 점은 뜻이 비슷한 청크임.
질문을 적으면 별 표시가 찍히고, 가져온 청크와 선으로 이어짐.
- 같은 문서의 청크끼리 모여 있는가. 내 질문은 어느 문서 가까이에 떨어지는가

In [ ]:
lab.map_panel()

## 6. 찾기 실험(선택)
검색은 두 점수를 섞어 순위를 정함. **의미 점수**(벡터가 가까운 정도)와 **키워드 점수**(같은 단어가 나온 정도)임.
- 키워드 비중 0 은 의미만, 1 은 키워드만 씀. 0.3 근처가 무난한 출발점임
- 고유 명칭 · 숫자가 든 질문과, 일상어로 돌려 말한 질문을 각각 넣어 순위가 어떻게 바뀌는지 봄

In [ ]:
lab.search_panel()

## 7. 문서를 붙여 답하기
같은 질문을 문서 없이 한 번, 찾은 청크를 붙여 한 번 물어 나란히 봄. 아래쪽 작은 글씨에서 근거 문서 · 입력 토큰 · 응답 시간을 확인함.
- '모델에 들어간 입력 보기'를 켜면 모델이 실제로 받은 글 전체가 보임. RAG 는 결국 **입력에 문서를 끼워 넣는 일**임

In [ ]:
lab.ask_panel()

## 8. 설정 실험
바꿀 설정 하나를 고르고 값 목록을 적으면, 확인 질문 전체를 값마다 다시 풀어 **답 점수 · 검색 적중 · 응답 시간**을 한 그림에 그림.
- 실행 전에 어느 값이 가장 좋을지 먼저 예측해 봄
- 점수만 보지 않음. 점수가 같으면 더 빠르고 입력이 짧은 설정이 좋은 설정임
- 서로 다른 설정을 **둘 이상** 실험함(예 : 붙일 청크 수, 청크 크기)

In [ ]:
lab.experiment_panel()

## 9. 모델 겨루기(선택)
1단계 조절판으로 돌아가 모델을 하나 더 올린 뒤, 같은 질문과 같은 근거 문서를 주고 답과 속도를 비교함.
- 작은 모델(`Qwen3-0.6B`)과 기본 모델의 차이, 다른 계열 모델(`Qwen3-8B`)과의 차이를 봄
- GPU 메모리가 넉넉하면 `Qwen3-14B` 같은 큰 모델도 올릴 수 있음(처음 한 번 내려받음)
- 8단계 조절판에서 바꿀 설정을 '모델'로 고르면 확인 질문 점수로도 비교할 수 있음

In [ ]:
lab.arena_panel()

## 10. 내 챗봇
지금까지 정한 설정으로 대화함. 말투 · 역할 칸에 지시를 적으면 답하는 방식이 바뀜.
충분히 써 본 뒤 아래 셀로 **내 솔루션**을 저장함. 7회차는 이 솔루션에서 이어 감.

In [ ]:
lab.chat()

In [ ]:
lab.save_solution()

## 11. 판단 기록 · 결과 묶기
오늘 실험에서 **무엇을 어떤 값으로 정했고 왜 그렇게 정했는지** 적음. 기록표와 보고서(`my/report.md`)가 함께 만들어짐.
끝으로 결과 묶기를 실행하고, 왼쪽 파일 창에서 `my_results_*.zip`을 오른쪽 단추 → Download 로 내려받음.

In [ ]:
lab.note_panel()

In [ ]:
lab.pack()

---
### 더 해 보기
- 문서를 더 넣거나(2단계) 일부러 **틀린 문서**를 넣고 답이 어떻게 바뀌는지 봄 : 모델은 붙여 준 문서를 믿음
- 기준일이 다른 문서 두 건을 넣고 어느 것을 근거로 답하는지 봄
- 조절판 없이 직접 부르기 : `lab.ask("질문", k=2)`, `lab.experiment("size", [200, 400, 800])`, `lab.board()`
- 5회차의 도메인 과제(JSON 답 채점)를 이 서버에서 재려면 : `lab.sh("python src/my_start.py --domain G")` 뒤 `lab.sh("python src/benchmark.py --name baseline --limit 10")`